# LUMYVUE V6 — Run 02 evidence notebook (read-only)

**What this is.** A read-only record of what the `open_run02` run produced: training curves, selection results, evaluation tables, the V5 vs V6 comparison, the export checks and the gate board. Every number below is read from files the run saved (`astra_runs/open_run02/`). Nothing is trained, recomputed from images, or written back to the run folder, and the locked test split is never touched.

**Honest reading rules (carry these into the thesis).**
- The run is **development-stage**. The locked test split was **not evaluated** (it has no expert labels and about 85 % of it was in V5's training data).
- Labels are **researcher-reviewed**, partly AI-assisted (recorded as such). They are not dermatologist-verified.
- Fusion-split metrics are out-of-fold but mostly on images V5 also trained on, so they are **not** a V5 comparison. The V5 comparison uses validation images V5 never trained on (section 8).
- Results are a single seed (42). Intervals are wide because the fair set has 132 images (47 hyperpigmentation positives, 26 all-negative, 18 eczema positives).

In [ ]:
import json, os, glob
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

CANDIDATES = [os.environ.get("LUMYVUE_RUN_DIR", ""),
              "/mnt/c/Users/Bautista/source/repos/SkinDevApp/v6/astra_runs/open_run02",
              r"C:\Users\Bautista\source\repos\SkinDevApp\v6\astra_runs\open_run02",
              "/mnt/c/Users/Bautista/source/repos/SkinDevApp/v6/backups/open_run02_complete_20261102_1715",
              r"C:\Users\Bautista\source\repos\SkinDevApp\v6\backups\open_run02_complete_20261102_1715"]
RUN = next((Path(c) for c in CANDIDATES if c and (Path(c) / "metrics").is_dir()), None)
assert RUN is not None, "run folder not found: set LUMYVUE_RUN_DIR"
M = RUN / "metrics"
print("reading from:", RUN)

## 1. Run identity and integrity

In [ ]:
fs = json.load(open(RUN / "report" / "final_summary.json"))
show = {k: v for k, v in fs.items() if k != "unit_tests"}
print(json.dumps(show, indent=1))
ut = fs["unit_tests"]; print(f"\nunit tests: ran {ut['ran']}, passed {ut['passed']}, failures {len(ut['failures'])}, errors {len(ut['errors'])}, skipped {len(ut['skipped'])}")
for s in ut["skipped"]: print("  skipped:", s[0].split(" ")[0], "-", s[1])
print("\ngate board:"); display(pd.read_csv(RUN / "report" / "stage_gate_board.csv"))

## 2. Global model — D (ImageNet start) vs E (V5 backbone start)
Phase A trains the new head with the backbone frozen; phase B fine-tunes from layer 100 with batch-norm frozen. The vertical line marks the A→B boundary. Selection used macro AP on validation images V5 never trained on (section 8 / table below).

In [ ]:
def hist(exp):
    a = pd.read_csv(RUN / "checkpoints" / exp / "seed_42" / "phaseA_history.csv"); a["phase"] = "A"
    b = pd.read_csv(RUN / "checkpoints" / exp / "seed_42" / "phaseB_history.csv"); b["phase"] = "B"
    d = pd.concat([a, b], ignore_index=True); d["step"] = np.arange(1, len(d) + 1); return d, len(a)
H = {e: hist(e) for e in ["D_multilabel3", "E_v5_backbone_multilabel3"]}
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for e, (d, na) in H.items():
    ax[0].plot(d.step, d.loss, label=f"{e} train"); ax[0].plot(d.step, d.val_loss, "--", label=f"{e} val")
    ax[1].plot(d.step, d.val_macro_ap, label=e); ax[2].plot(d.step, d.val_macro_f1, label=e)
for a_, t in zip(ax, ["loss (masked BCE)", "validation macro AP", "validation macro F1 @0.5"]):
    a_.axvline(na + 0.5, color="grey", ls=":"); a_.set_title(t); a_.set_xlabel("epoch (A then B)"); a_.legend(fontsize=7)
plt.tight_layout(); plt.show()
rows = []
for e, (d, na) in H.items():
    i = d.val_macro_ap.idxmax(); rows.append(dict(experiment=e, epochs_A=na, epochs_B=len(d) - na, best_step=int(d.step[i]), best_phase=d.phase[i], best_val_macro_ap=d.val_macro_ap[i], val_macro_f1_at_best=d.val_macro_f1[i]))
display(pd.DataFrame(rows).round(4))

In [ ]:
print("D vs E on validation images, split by V5 exposure (per condition):")
x = pd.read_csv(M / "d_vs_e_validation_by_exposure.csv"); display(x[["subset", "experiment", "condition", "pos", "neg", "precision", "recall", "f1", "ap"]].round(3))
print("selected:", json.load(open(M / "global_selection.json")).get("selected"))

## 3. ROI MobileNetV2 (4 classes: acne, hyperpigmentation, eczema, background false positive)

In [ ]:
ra = pd.read_csv(RUN / "checkpoints/ROI/seed_42/phaseA_history.csv"); rb = pd.read_csv(RUN / "checkpoints/ROI/seed_42/phaseB_history.csv")
r = pd.concat([ra.assign(phase="A"), rb.assign(phase="B")], ignore_index=True); r["step"] = np.arange(1, len(r) + 1)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(r.step, r.loss, label="train"); ax[0].plot(r.step, r.val_loss, "--", label="val"); ax[0].set_title("loss")
ax[1].plot(r.step, r.accuracy, label="train"); ax[1].plot(r.step, r.val_accuracy, "--", label="val"); ax[1].set_title("accuracy")
ax[2].plot(r.step, r.val_macro_f1, label="val macro F1"); ax[2].plot(r.step, r.val_macro_recall, "--", label="val macro recall"); ax[2].set_title("validation macro F1 / recall")
for a_ in ax: a_.axvline(len(ra) + 0.5, color="grey", ls=":"); a_.set_xlabel("epoch (A then B)"); a_.legend(fontsize=7)
plt.tight_layout(); plt.show()
print(json.dumps(json.load(open(M / "roi_training_summary.json")), indent=1)[:1500])
print("\nROI validation per class:"); display(pd.read_csv(M / "roi_validation_per_class.csv").round(3))

## 4. Fusion models — 7 variants × (5 grouped folds + final fit)
Every fold and the final fit stop by early stopping. The table lists the epochs each used and its last validation loss (fold sizes are small, so curves are noisy).

In [ ]:
VARS = ["full", "global_yolo", "global_only", "roi_only", "no_acne_evidence", "no_eczema_evidence", "no_hyper_evidence"]
def fold_hist(var, fold):
    fs = sorted(glob.glob(str(RUN / "fusion" / "recovery" / var / "seed_42" / fold / "attempt_*" / "epoch_*.history.csv")))
    return pd.read_csv(fs[-1]) if fs else None
rows = []
for v in VARS:
    for f in ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4", "final"]:
        h = fold_hist(v, f)
        if h is not None: rows.append(dict(variant=v, fit=f, epochs=len(h), best_val_loss=h.val_loss.min(), last_val_loss=h.val_loss.iloc[-1]))
FT = pd.DataFrame(rows); print(f"{len(FT)} fits found (expected 42)")
display(FT.pivot(index="variant", columns="fit", values="epochs").reindex(VARS))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for f in ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]:
    h = fold_hist("full", f); ax[0].plot(h.epoch, h.val_loss, label=f)
ax[0].set_title("variant 'full': validation loss per fold"); ax[0].legend(fontsize=7); ax[0].set_xlabel("epoch")
FT.groupby("variant").best_val_loss.mean().reindex(VARS).plot.barh(ax=ax[1]); ax[1].set_title("mean best validation loss over fits (lower is better)")
plt.tight_layout(); plt.show()

## 5. Development evaluation (fusion split, out-of-fold)
Out-of-fold means each image was scored by a model that did not train on it. **This is not a V5 comparison and not a test result.** About 85 % of these images were in V5's training data.

In [ ]:
names = {"global_only": "Global only (recalibrated)", "global_yolo": "Global + YOLO evidence", "full": "Full fusion (+ROI)", "roi_only": "ROI branch only"}
T = pd.concat([pd.read_csv(M / f"development_{k}_per_condition.csv").assign(variant=v) for k, v in names.items()], ignore_index=True)
display(T.pivot(index="variant", columns="condition", values="f1").round(3))
ax = T.pivot(index="condition", columns="variant", values="f1").plot.bar(figsize=(9, 4), rot=0); ax.set_ylabel("F1 (fusion split, out-of-fold)"); ax.set_ylim(0, 1); plt.tight_layout(); plt.show()
print("per-condition detail:"); display(T[["variant", "condition", "n_evaluable", "pos", "neg", "precision", "recall", "specificity", "f1"]].round(3))

In [ ]:
s = json.load(open(M / "development_oof_summary.json"))
fn = s["final_normal"]; s1 = s["stage1_normal"]
print(f"Derived-Normal decision (fusion split, n={s['n']}):")
print(f"  Stage 1 accepted {s1['accepted_n']} of {s1['reference_n']} references as Normal-decidable; abstention rate {s1['abstention_rate']:.1%}")
print(f"  Final (after Stage 2) accepted {fn['accepted_n']}; abstention rate {fn['abstention_rate']:.1%}; false Normal {fn['false_normal']}; recall over ALL references {fn['recall_all_references']:.1%}")
print("  A low error on accepted cases must be read together with this abstention rate; V5 never abstains.")

## 6. Calibration of the fusion variants (out-of-fold)

In [ ]:
c = pd.read_csv(M / "fusion_calibration_oof.csv"); display(c.round(4).head(30))

## 7. Stage rules

In [ ]:
display(pd.read_csv(M / "development_oof_stage_per_condition.csv").round(3))

## 8. V5 vs V6 on validation images V5 never trained on  ← the thesis comparison
**Set:** validation images not in V5's training data (by id, MD5 and leakage group), fully verified labels. **Systems:** V5 argmax (its own decision rule), V5 with thresholds tuned on the fusion split (A2), Global E, Global+YOLO (E), Full fusion (F), ROI only, Global recalibrated. **Thresholds** come from the fusion split only. **Intervals** are 95 % bootstrap over leakage groups; an interval that includes 0 means the data do not show a difference (it does not show equivalence). **Caveat:** validation was used to select Global E over D and to stop ROI training, so V6 may look slightly better here than on fresh photos; V5 was not tuned on it. Abstention (system G) is not in this table.

In [ ]:
fair = pd.read_csv(M / "v5_vs_v6_fair_set.csv"); pair = pd.read_csv(M / "v5_vs_v6_fair_set_paired.csv")
display(fair.round(3)); display(pair.round(3))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a_, met, title in zip(ax, ["macro_f1", "hp_missed_rate", "false_alarm_rate"], ["macro F1 difference vs V5 (higher = better)", "hyperpigmentation missed — difference vs V5 (lower = better)", "clean photo flagged hyperpigmentation — difference vs V5 (lower = better)"]):
    p = pair[pair.metric == met]; y = np.arange(len(p))
    a_.errorbar(p.difference, y, xerr=[p.difference - p.ci_lo, p.ci_hi - p.difference], fmt="o", capsize=4); a_.axvline(0, color="grey"); a_.set_yticks(y); a_.set_yticklabels(p.system, fontsize=8); a_.set_title(title, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Pre-registered criterion (v6/THESIS_TRACKER.md, fixed before results): V6 = Global + YOLO fusion counts as an improvement only if
# (a) hyperpigmentation-missed AND clean-flagged are each not worse than V5, and at least one is better with a 95% interval that excludes zero, and
# (b) Acne and Eczema F1 are each not more than 0.05 below V5.
f = fair.set_index("system"); A = f.loc["A_v5_argmax"]
for name in ["E_global_plus_yolo", "F_full_fusion"]:
    E = f.loc[name]; P = pair[(pair.system == name) & (pair.metric.isin(["hp_missed_rate", "false_alarm_rate"]))]
    a_notworse = bool(E.hp_missed_as_no_condition <= A.hp_missed_as_no_condition and E.all_neg_flagged_hp <= A.all_neg_flagged_hp)
    a_sig = bool(((P.difference < 0) & P.excludes_zero).any())
    b_ok = bool(E.f1_acne >= A.f1_acne - 0.05 and E.f1_eczema >= A.f1_eczema - 0.05)
    print(f"{name}: (a) not worse on both errors = {a_notworse}; one better with CI excluding 0 = {a_sig}; (b) acne/eczema F1 within 0.05 = {b_ok}  ->  {'MET' if (a_notworse and a_sig and b_ok) else 'NOT MET: report as no clear improvement'}")
    print(f"    hyper missed {int(E.hp_missed_as_no_condition)} vs V5 {int(A.hp_missed_as_no_condition)} | clean flagged {int(E.all_neg_flagged_hp)} vs {int(A.all_neg_flagged_hp)} | eczema F1 {E.f1_eczema:.3f} vs {A.f1_eczema:.3f}")

### 8b. Eczema on the same fair set (post-hoc diagnostic — a hypothesis, not a result)
Only 18 eczema-positive photos. The V6 models *rank* eczema better than V5 (higher AP/AUC) but lose at the calibrated cutoff (thresholds were set on only 33 eczema positives in the fusion split). No threshold was re-tuned on these photos.

In [ ]:
e = pd.read_csv(M / "eczema_fair_set_diagnostic.csv"); print(e.groupby("truth").size().to_dict())
pos = e[e.truth == "eczema"]; print(f"eczema positives {len(pos)}: V5 found {int(pos.v5_says.sum())}, full fusion found {int(pos.full_says.sum())}")
display(e[(e.v5_says == (e.truth == "eczema")) & (e.full_says != (e.truth == "eczema"))])

## 9. Export and ONNX faithfulness
The notebook's own check compared Keras on the **GPU** with ONNX on the **CPU** and flagged everything outside 1e-4. A follow-up compared the same images three ways: the gap is GPU-vs-CPU numeric noise, not an export defect.

In [ ]:
ex = json.load(open(RUN / "export" / "v6_export_status.json")); print({k: v.get("status") for k, v in ex.items()})
v = json.load(open(RUN / "export" / "v6_keras_vs_onnx_verification.json"))
display(pd.DataFrame([dict(component=k, status=x.get("status"), max_abs_diff=max(x.get("max_abs_diff_per_output", [np.nan])), within_1e_4=x.get("within_tolerance")) for k, x in v.items() if isinstance(x, dict)]).round(5))
pc = RUN / "export" / "v6_global_features_parity_check.csv"
if pc.exists(): print("Global feature model, 160 validation photos x 3 conditions (Keras GPU / Keras CPU / ONNX CPU):"); display(pd.read_csv(pc).round(5))
else: print("(follow-up parity table not found in this folder)")

## 10. What this run does and does not show
- **Does show:** the pipeline trains end to end and reproduces (audit chain intact, originals unchanged, 35 of 37 unit tests pass with 2 skipped by design); components export to ONNX and match Keras on CPU.
- **Does not show:** that V6 beats V5. On validation images V5 never trained on, the pre-registered criterion was **not met** (section 8). Global E and the fusion models rank conditions somewhat better than V5, and acne F1 is higher, but the error rates the thesis targets are not significantly better and eczema F1 fell below the allowed margin.
- **Not done:** the locked test (no expert labels), an independent labelled set, dermatologist validation, C# parity, multi-seed variability.